# Inferencia Causal con LLaMA

Este programa permite realizar inferencia causal sobre el LLM de código abierto LLaMA de 3B parámetros, y pre-entrenado sobre 1T tokens.

La inferencia causal en modelos de lenguaje  permite entender las relaciones causa-efecto entre variables (aka. CausalLM). Esto se puede aplicar para responder (efecto) una pregunta (causa), para establecer entailment entre una oración (causa) y otra (efecto), etc.

En este ejemplo, se utilizará el modelo causal para responder preguntas. El modelo LLaMA puede utilizar su  propio tokenizador (*LLamaTokenizer*) o bien el método genérico *AutoTokenizer*:

Instalamos algunos paquetes de transformers y facilidades para aceleración de hardware:

In [1]:
!pip install transformers
!pip install sentencepiece
!pip install accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 121.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninst

Importamos algunas bibliotecas para transformers:

In [2]:
from transformers import LlamaTokenizer, LlamaForCausalLM, AutoTokenizer
from transformers import TrainingArguments, AutoModelForSequenceClassification
import torch
import numpy as np

Definimos la función **CargarModelo(NombreModelo)** para cargar el tokenizador (*LLamaTokenizer*) y el modelo pre-entrenado basado en **NombreModelo**, retorna el modelo y tokenizador cargados:

In [3]:
def CargarModelo(NombreModelo):
   tokenizador = LlamaTokenizer.from_pretrained(NombreModelo)
   modelo = LlamaForCausalLM.from_pretrained(
       NombreModelo, torch_dtype=torch.float16, device_map='auto')
   return(modelo,tokenizador)

Definimos la función **CodificarPrompt(prompt)*¨*, que tokeniza un string que indica el **prompt** y retorna los IDs de cada token que lo compone:


In [4]:
def CodificarPrompt(prompt,tokenizador):
  inputIDs = tokenizador(prompt, return_tensors="pt").input_ids
  return(inputIDs)

Iniciamos el programa principal, cargando el modelo, definiendo el prompt a enviar al modelo, y generando la salida con un número máximo de tokens:

In [5]:
# Otros modelos LLaMA:
#     'openlm-research/open_llama_7b' (7B parámetros)
#     'openlm-research/open_llama_13b' (13B parámetros)
(modelo,tokenizador) = CargarModelo('openlm-research/open_llama_3b')
# El prompt debe comenzar con Q (question) y terminar con A (Answer)
# de modo que el modelo realice  la inferencia que continua a A
prompt   = 'Q: ¿Cuál es el animal más grande en Chile?\nA:'
inputIDs = CodificarPrompt(prompt,tokenizador)
# Mover el tensor inputIDs a la misma GPU que el modelo
inputIDs = inputIDs.to(modelo.device)
salida   = modelo.generate(input_ids=inputIDs, max_new_tokens=20)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/593 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/534k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message


config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/6.85G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.85G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Note que el modelo genera una respuesta de salida con los IDs de cada token, por lo que estos se deben decodificar (*decode*) para convertirlos a palabras:

In [6]:
print(tokenizador.decode(salida[0]))

<s>Q: ¿Cuál es el animal más grande en Chile?
A: El lobo.
Q: ¿Cuál es el animal más pequeño


In [7]:
prompt   = 'Q: ¿Cuál es el animal más pequeño en Chile?\nA:'
inputIDs = CodificarPrompt(prompt,tokenizador)
# Mover el tensor inputIDs a la misma GPU que el modelo
inputIDs = inputIDs.to(modelo.device)
salida   = modelo.generate(input_ids=inputIDs, max_new_tokens=40)
print(tokenizador.decode(salida[0]))

<s>Q: ¿Cuál es el animal más pequeño en Chile?
A: El pájaro más pequeño en Chile es el pájaro de la tarde (Tachybaptus rufolavatus).

A: El p
